In [ ]:
# 2월 15일 11시 8분 시작

# Notebook 02 (쿼리 3유형 생성 + leakage 판정 + 최대 3회 재생성 + 저장/체크포인트) “완성본”이다.
너가 이미 01에서 만든 2nd_exp/data/{year}/sampled_patents.csv와 meta.json을 그대로 읽어서 동작한다. 또한 기존 코드에서 쓰던 방식대로 Ollama(localhost:11434) + gpt-oss 호출 구조를 유지한다.

전체 구조 (Notebook 02가 하는 일)
입력: 2nd_exp/data/{year}/sampled_patents.csv (연도별 1050개)
출력:
2nd_exp/queries/{year}/queries_all.csv (type 포함, leakage/regen 로그 포함)
2nd_exp/queries/{year}/queries_by_type/{type}.csv
2nd_exp/queries/{year}/stats_querygen.json (요약 통계)

중간 저장(체크포인트): 2nd_exp/queries/{year}/checkpoints/*.csv

In [1]:
# =========================================
# Notebook 02 — Query Generation (3 types) + Leakage Filtering (3-gram overlap)
#  - Uses Ollama local server for gpt-oss generation (same as uploaded legacy code)
# =========================================
# Reads:
#   2nd_exp/data/{year}/sampled_patents.csv  (from Notebook 01)
# Writes:
#   2nd_exp/data/{year}/queries_final.csv
#   2nd_exp/data/{year}/queries.csv                  (compat alias)
#   2nd_exp/cache/query_gen/{year}/raw_generations.jsonl
#   2nd_exp/cache/query_gen/{year}/checkpoints/*.csv (batch checkpoints)
#   2nd_exp/config/notebook02_config.json
#   2nd_exp/logs/notebook02_log.txt
#
# Decisions implemented (B1~B6):
#  - years: 2005/2015/2025, processed together
#  - query types: summary / keyword-heavy / function-oriented
#  - LLM: gpt-oss via Ollama (http://localhost:11434/api/generate)
#  - prompt templates fixed per type (appendix-ready)
#  - query length: 20~50 words
#  - leakage: 3-gram overlap vs (claims+abstract), stopword removal + stemming
#  - threshold: ANY overlap (>=1) triggers leakage
#  - regeneration: up to 3 attempts if leakage or length out of bounds
#  - randomness fixed: derived per (seed, year, patent_id, type, attempt)
#  - all artifacts saved under 2nd_exp/
# =========================================

import os
import re
import json
import time
import random
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import requests

# ----------------------------
# 0) Global settings
# ----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

YEARS = [2005, 2015, 2025]
QUERY_TYPES = ["summary", "keyword", "function"]

MIN_WORDS = 20
MAX_WORDS = 50
MAX_REGEN = 3

# leakage rule: any 3-gram overlap triggers leakage
LEAK_OVERLAP_MIN_COUNT = 1

# Ollama / gpt-oss (matching legacy code style)
OLLAMA_API_URL = "http://localhost:11434/api/generate"
MODEL_NAME = "gpt-oss:20b"

# reproducibility parameters (fixed)
TEMPERATURE = 0.0
TOP_P = 1.0

# batch checkpointing
SAVE_EVERY_N = 50   # save checkpoint every N patents per year

BASE_DIR = "2nd_exp"
DATA_DIR = os.path.join(BASE_DIR, "data")
CACHE_DIR = os.path.join(BASE_DIR, "cache", "query_gen")
CFG_DIR = os.path.join(BASE_DIR, "config")
LOG_DIR = os.path.join(BASE_DIR, "logs")

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(CFG_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

LOG_PATH = os.path.join(LOG_DIR, "notebook02_log.txt")

def log(msg: str):
    print(msg)
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(msg + "\n")

# reset log
with open(LOG_PATH, "w", encoding="utf-8") as f:
    f.write("")

log("=== Notebook 02 started ===")
log(f"SEED={SEED}, YEARS={YEARS}, TYPES={QUERY_TYPES}")
log(f"WORD_LIMIT={MIN_WORDS}..{MAX_WORDS}, MAX_REGEN={MAX_REGEN}")
log(f"LEAK_RULE: 3-gram overlap_count >= {LEAK_OVERLAP_MIN_COUNT} => leakage")
log(f"Ollama: url={OLLAMA_API_URL}, model={MODEL_NAME}")
log(f"LLM params: temperature={TEMPERATURE}, top_p={TOP_P}")
log(f"Working dir: {os.getcwd()}")

# ----------------------------
# 1) Stopwords + stemming (robust fallback)
# ----------------------------
DEFAULT_STOPWORDS = set("""
a an and are as at be but by for from has have he her hers him his i if in into is it its
me my of on or our ours she that the their theirs them they this to was we were what when where which who why will with you your yours
""".split())

# Try nltk stemmer if available; fallback to a simple stemmer
try:
    from nltk.stem import PorterStemmer
    _porter = PorterStemmer()
    def stem_token(tok: str) -> str:
        return _porter.stem(tok)
    STEMMER_NAME = "nltk.PorterStemmer"
except Exception:
    def stem_token(tok: str) -> str:
        tok = tok.lower()
        for suf in ["ingly","edly","ing","ed","ly","es","s"]:
            if tok.endswith(suf) and len(tok) > len(suf) + 2:
                return tok[:-len(suf)]
        return tok
    STEMMER_NAME = "fallback_suffix_stripper"

def tokenize_for_overlap(text: str) -> List[str]:
    text = str(text).lower()
    toks = re.findall(r"[a-z0-9]+", text)
    toks = [t for t in toks if t not in DEFAULT_STOPWORDS]
    toks = [stem_token(t) for t in toks]
    toks = [t for t in toks if len(t) > 1]
    return toks

def make_ngrams(tokens: List[str], n: int = 3) -> List[Tuple[str, ...]]:
    if len(tokens) < n:
        return []
    return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

# ----------------------------
# 2) Ollama connectivity test (same spirit as legacy code)
# ----------------------------
def test_ollama_connection() -> bool:
    try:
        resp = requests.get("http://localhost:11434/api/tags", timeout=5)
        if resp.status_code != 200:
            log(f"✗ Ollama /api/tags returned status={resp.status_code}")
            return False
        models = resp.json().get("models", [])
        model_names = [m.get("name","") for m in models]
        log("✓ Ollama is running")
        log(f"✓ Available models: {', '.join(model_names[:20])}" + (" ..." if len(model_names) > 20 else ""))
        if (MODEL_NAME not in model_names) and (MODEL_NAME.replace(":", "-") not in model_names):
            log(f"⚠ Warning: model '{MODEL_NAME}' not found in tags list (it may still load on demand).")
        return True
    except requests.exceptions.ConnectionError:
        log("✗ Cannot connect to Ollama. Please make sure Ollama is running: `ollama serve`")
        return False
    except Exception as e:
        log(f"✗ Error testing Ollama: {repr(e)}")
        return False

OLLAMA_OK = test_ollama_connection()
if not OLLAMA_OK:
    raise RuntimeError("Ollama is not available. Start Ollama first, then rerun Notebook 02.")

# ----------------------------
# 3) Prompt templates (fixed per type; appendix-ready)
# ----------------------------
PROMPTS = {
    "summary": (
        "You are generating a patent prior-art search query.\n"
        "Write ONE English query (not a list), 20–50 words.\n"
        "Goal: summarize the invention at a high level (problem, novelty, main mechanism) WITHOUT copying phrases.\n"
        "Do NOT quote or copy any exact phrases from the input.\n"
        "Return only the query text.\n\n"
        "[INPUT]\n"
        "ABSTRACT:\n{abstract}\n\n"
        "CLAIMS:\n{claims}\n"
    ),
    "keyword": (
        "You are generating a patent prior-art search query.\n"
        "Write ONE English query (not a list), 20–50 words.\n"
        "Goal: include components, technical keywords, materials/structures, and domain-specific terms.\n"
        "Avoid direct copying; PARAPHRASE.\n"
        "Return only the query text.\n\n"
        "[INPUT]\n"
        "ABSTRACT:\n{abstract}\n\n"
        "CLAIMS:\n{claims}\n"
    ),
    "function": (
        "You are generating a patent prior-art search query.\n"
        "Write ONE English query (not a list), 20–50 words.\n"
        "Goal: focus on functions, operations, and what the invention enables (use-case, effects, functional constraints).\n"
        "Avoid direct copying; PARAPHRASE.\n"
        "Return only the query text.\n\n"
        "[INPUT]\n"
        "ABSTRACT:\n{abstract}\n\n"
        "CLAIMS:\n{claims}\n"
    ),
}

def add_forbidden_ngrams_instruction(base_prompt: str, forbidden_ngrams: List[Tuple[str,...]]) -> str:
    if not forbidden_ngrams:
        return base_prompt
    phrases = [" ".join(ng) for ng in forbidden_ngrams[:25]]
    forb = "\n".join([f"- {p}" for p in phrases])
    rule = (
        "\n\n[FORBIDDEN PHRASES]\n"
        "Do NOT use any of the following exact 3-word phrases in your query:\n"
        f"{forb}\n"
    )
    return base_prompt + rule

# ----------------------------
# 4) Leakage scoring + word count + output cleaning
# ----------------------------
def leakage_score_3gram(source_text: str, query_text: str) -> Dict[str, object]:
    src_toks = tokenize_for_overlap(source_text)
    qry_toks = tokenize_for_overlap(query_text)

    src_ngr = set(make_ngrams(src_toks, 3))
    qry_set = set(make_ngrams(qry_toks, 3))

    inter = sorted(list(qry_set.intersection(src_ngr)))
    overlap_count = len(inter)
    denom = max(1, len(qry_set))
    overlap_ratio = overlap_count / denom

    return {
        "overlap_count": overlap_count,
        "query_ngram_count": len(qry_set),
        "overlap_ratio": overlap_ratio,
        "overlapping_ngrams": inter
    }

def word_count(text: str) -> int:
    return len(re.findall(r"\b\w+\b", str(text)))

def clean_llm_output(text: str) -> str:
    s = str(text).strip()
    s = re.sub(r"^[-*•]+\s*", "", s)
    # remove common prefixes (legacy code did similar)
    prefixes_to_remove = [
        "Search Query:", "Query:", "Here is", "The search query is:",
        "A possible search query:", "Generated query:", "Here's",
        "The query:", "Suggested query:"
    ]
    for p in prefixes_to_remove:
        if s.lower().startswith(p.lower()):
            s = s[len(p):].strip()
    s = s.strip().strip('"').strip("'").strip()
    s = re.sub(r"\s+", " ", s).strip()
    return s

# ----------------------------
# 5) gpt-oss call via Ollama (based on legacy code)
# ----------------------------
def call_gpt_oss(prompt: str, seed: int, temperature: float, top_p: float = 1.0, timeout: int = 180) -> str:
    """
    Ollama /api/generate call in the same style as legacy code.
    We attempt to pass seed/top_p via 'options' (Ollama supports options; if ignored, still works).
    """
    payload = {
        "model": MODEL_NAME,
        "prompt": prompt,
        "temperature": float(temperature),
        "top_p": float(top_p),
        "stream": False,
        # options is commonly supported; if your Ollama build ignores it, results may still vary slightly.
        "options": {
            "seed": int(seed),
            "top_p": float(top_p),
        }
    }
    resp = requests.post(OLLAMA_API_URL, json=payload, timeout=timeout)
    if resp.status_code != 200:
        raise RuntimeError(f"Ollama API error: {resp.status_code} / {resp.text[:200]}")
    data = resp.json()
    # legacy code used response.json()['response']
    return str(data.get("response", "")).strip()

# ----------------------------
# 6) Load sampled patents (force dtype=str including patent_id)
# ----------------------------
def _normalize_patent_id(series: pd.Series, year: int) -> pd.Series:
    pid = series.astype(str).str.strip()
    pid = pid.str.replace(r"\.0$", "", regex=True)
    bad = pid.isna() | (pid.str.len() == 0) | (pid.str.lower().isin(["nan","none"]))
    if bad.any():
        raise ValueError(f"[{year}] Found invalid patent_id after normalization. Fix upstream in Notebook 01.")
    return pid

def load_sampled_patents(year: int) -> pd.DataFrame:
    path = os.path.join(DATA_DIR, str(year), "sampled_patents.csv")
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing sampled_patents.csv for year={year}: {path}")

    df = pd.read_csv(path, dtype=str, keep_default_na=False, na_values=[], low_memory=False)

    for col in ["patent_id", "claims", "abstract"]:
        if col not in df.columns:
            raise ValueError(f"[{year}] Required column missing: {col}. Found: {df.columns.tolist()}")

    df["patent_id"] = _normalize_patent_id(df["patent_id"], year)
    df["claims"] = df["claims"].fillna("").astype(str)
    df["abstract"] = df["abstract"].fillna("").astype(str)

    if "wipo" in df.columns:
        df["wipo"] = df["wipo"].astype(str).str.strip()
    else:
        df["wipo"] = ""

    return df

patents_by_year = {y: load_sampled_patents(y) for y in YEARS}
for y in YEARS:
    log(f"[{y}] sampled_patents loaded: rows={len(patents_by_year[y])}, cols={len(patents_by_year[y].columns)}")
log(f"Stemmer: {STEMMER_NAME}, Stopwords: {len(DEFAULT_STOPWORDS)} words")

# ----------------------------
# 7) Paths + resume support
# ----------------------------
def year_paths(year: int) -> Dict[str, str]:
    ydir = os.path.join(DATA_DIR, str(year))
    cdir = os.path.join(CACHE_DIR, str(year))
    ckpt = os.path.join(cdir, "checkpoints")
    os.makedirs(cdir, exist_ok=True)
    os.makedirs(ckpt, exist_ok=True)
    return {
        "year_dir": ydir,
        "cache_dir": cdir,
        "checkpoint_dir": ckpt,
        "raw_jsonl": os.path.join(cdir, "raw_generations.jsonl"),
        "queries_final": os.path.join(ydir, "queries_final.csv"),
        "queries_alias": os.path.join(ydir, "queries.csv"),
    }

def load_existing_done_set(path_queries_final: str) -> set:
    if not os.path.exists(path_queries_final):
        return set()
    df = pd.read_csv(path_queries_final, dtype=str, keep_default_na=False, na_values=[])
    df["patent_id"] = df["patent_id"].astype(str).str.strip()
    df["query_type"] = df["query_type"].astype(str).str.lower().str.strip()
    return set((r["patent_id"], r["query_type"]) for _, r in df.iterrows())

done_map = {y: load_existing_done_set(year_paths(y)["queries_final"]) for y in YEARS}
for y in YEARS:
    log(f"[{y}] resume: existing completed pairs = {len(done_map[y])}")

# ----------------------------
# 8) Raw logging helpers + checkpointing
# ----------------------------
def append_raw_jsonl(path: str, record: Dict):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

def save_checkpoint(year: int, outputs: List[Dict], suffix: str):
    paths = year_paths(year)
    df = pd.DataFrame(outputs)
    outp = os.path.join(paths["checkpoint_dir"], f"queries_checkpoint_{suffix}.csv")
    df.to_csv(outp, index=False, encoding="utf-8")
    log(f"[{year}] checkpoint saved -> {outp} (rows={len(df)})")

def finalize_year_outputs(year: int, outputs: List[Dict]):
    paths = year_paths(year)
    df_new = pd.DataFrame(outputs)

    if os.path.exists(paths["queries_final"]):
        df_old = pd.read_csv(paths["queries_final"], dtype=str, keep_default_na=False, na_values=[])
        df = pd.concat([df_old, df_new], ignore_index=True)
    else:
        df = df_new.copy()

    # enforce dtypes
    df["patent_id"] = df["patent_id"].astype(str).str.strip()
    df["query_type"] = df["query_type"].astype(str).str.lower().str.strip()

    # deduplicate by (patent_id, query_type), keep last
    df = df.drop_duplicates(subset=["patent_id", "query_type"], keep="last").reset_index(drop=True)

    df.to_csv(paths["queries_final"], index=False, encoding="utf-8")
    df.to_csv(paths["queries_alias"], index=False, encoding="utf-8")

    log(f"[{year}] queries_final saved -> {paths['queries_final']} (rows={len(df)})")
    log(f"[{year}] queries.csv alias saved -> {paths['queries_alias']} (rows={len(df)})")

# ----------------------------
# 9) Core generation: 3 types + leakage + length + regen
# ----------------------------
def generate_one_query(year: int, patent_row: pd.Series, qtype: str) -> Dict:
    pid = str(patent_row["patent_id"]).strip()
    claims = str(patent_row["claims"])
    abstract = str(patent_row["abstract"])
    wipo = str(patent_row.get("wipo", ""))

    source_text = abstract + "\n" + claims
    base_prompt = PROMPTS[qtype].format(abstract=abstract, claims=claims)

    forbidden = []
    last_err = ""
    last_gen = ""
    last_wc = 0
    last_leak = None

    for attempt in range(1, MAX_REGEN + 1):
        prompt = add_forbidden_ngrams_instruction(base_prompt, forbidden)

        # deterministic per (seed, year, pid, qtype, attempt)
        derived_seed = (abs(hash(f"{SEED}|{year}|{pid}|{qtype}|{attempt}")) % (2**31 - 1))

        t0 = time.time()
        try:
            raw = call_gpt_oss(prompt, seed=derived_seed, temperature=TEMPERATURE, top_p=TOP_P)
            gen = clean_llm_output(raw)
        except Exception as e:
            raw = ""
            gen = ""
            last_err = repr(e)

        dt = time.time() - t0
        wc = word_count(gen)

        leak = leakage_score_3gram(source_text, gen)
        leak_count = leak["overlap_count"]
        leak_ratio = leak["overlap_ratio"]
        overlaps = leak["overlapping_ngrams"]

        last_gen, last_wc, last_leak = gen, wc, leak

        append_raw_jsonl(year_paths(year)["raw_jsonl"], {
            "year": year,
            "patent_id": pid,
            "query_type": qtype,
            "attempt": attempt,
            "seed": derived_seed,
            "temperature": TEMPERATURE,
            "top_p": TOP_P,
            "elapsed_sec": dt,
            "word_count": wc,
            "leak_overlap_count": leak_count,
            "leak_overlap_ratio": leak_ratio,
            "overlap_ngrams": [" ".join(x) for x in overlaps[:60]],
            "prompt": prompt,
            "raw_response": raw,
            "clean_query": gen,
            "error": last_err
        })

        length_ok = (MIN_WORDS <= wc <= MAX_WORDS)
        leak_ok = (leak_count < LEAK_OVERLAP_MIN_COUNT)
        nonempty = (len(gen) > 0)

        if length_ok and leak_ok and nonempty:
            return {
                "year": year,
                "patent_id": pid,
                "wipo": wipo,
                "query_type": qtype,
                "query_text": gen,
                "attempt_used": attempt,
                "query_word_count": wc,
                "leak_overlap_count": leak_count,
                "leak_overlap_ratio": leak_ratio,
                "leak_query_ngram_count": leak["query_ngram_count"],
                "status": "ok"
            }

        # update forbidden list to overlaps (strong constraint)
        forbidden = overlaps

    # After MAX_REGEN attempts, return last attempt but mark failure
    return {
        "year": year,
        "patent_id": pid,
        "wipo": wipo,
        "query_type": qtype,
        "query_text": last_gen,
        "attempt_used": MAX_REGEN,
        "query_word_count": last_wc,
        "leak_overlap_count": last_leak["overlap_count"] if last_leak else -1,
        "leak_overlap_ratio": last_leak["overlap_ratio"] if last_leak else -1.0,
        "leak_query_ngram_count": last_leak["query_ngram_count"] if last_leak else -1,
        "status": "failed_constraints"
    }

# ----------------------------
# 10) Main loop (3 years together)
# ----------------------------
all_outputs = {y: [] for y in YEARS}

for year in YEARS:
    dfp = patents_by_year[year].copy()
    done_pairs = done_map[year]

    log(f"\n[{year}] Begin generation for {len(dfp)} patents.")
    start_year = time.time()

    processed = 0
    new_rows = 0

    for _, row in dfp.iterrows():
        pid = str(row["patent_id"]).strip()

        for qtype in QUERY_TYPES:
            if (pid, qtype) in done_pairs:
                continue

            out = generate_one_query(year, row, qtype)
            all_outputs[year].append(out)
            new_rows += 1

        processed += 1

        if processed % SAVE_EVERY_N == 0:
            save_checkpoint(year, all_outputs[year], suffix=f"p{processed}")
            log(f"[{year}] progress: patents={processed}/{len(dfp)} new_rows={new_rows}")

    finalize_year_outputs(year, all_outputs[year])
    log(f"[{year}] Completed in {(time.time()-start_year)/60:.2f} min. newly_generated_rows={new_rows}")

# ----------------------------
# 11) Save config (reproducibility)
# ----------------------------
cfg = {
    "seed": SEED,
    "years": YEARS,
    "query_types": QUERY_TYPES,
    "word_limit": {"min_words": MIN_WORDS, "max_words": MAX_WORDS},
    "max_regen": MAX_REGEN,
    "leakage": {
        "source_text": "claims + abstract",
        "preprocess": {
            "stopwords": "DEFAULT_STOPWORDS (fixed list)",
            "stemming": STEMMER_NAME
        },
        "rule": "3-gram overlap",
        "threshold": f"overlap_count >= {LEAK_OVERLAP_MIN_COUNT} => leakage",
        "regen_forbidden_rule": "forbid overlapping 3-grams in regeneration prompt"
    },
    "llm": {
        "runtime": "ollama",
        "api_url": OLLAMA_API_URL,
        "model": MODEL_NAME,
        "temperature": TEMPERATURE,
        "top_p": TOP_P,
        "seed_strategy": "derived_seed = hash(SEED|year|patent_id|query_type|attempt)",
        "timeout_sec": 180
    },
    "checkpoint": {"save_every_n_patents": SAVE_EVERY_N},
    "outputs": {
        "queries_final": "2nd_exp/data/{year}/queries_final.csv",
        "queries_alias": "2nd_exp/data/{year}/queries.csv",
        "raw_generations": "2nd_exp/cache/query_gen/{year}/raw_generations.jsonl",
        "checkpoints": "2nd_exp/cache/query_gen/{year}/checkpoints/*.csv"
    }
}

cfg_path = os.path.join(CFG_DIR, "notebook02_config.json")
with open(cfg_path, "w", encoding="utf-8") as f:
    json.dump(cfg, f, indent=2, ensure_ascii=False)

log(f"\nSaved config -> {cfg_path}")
log("=== Notebook 02 completed ===")

print("\nDone. Next: run Notebook 03 (evaluation over 12 models).")
print("Queries saved under: 2nd_exp/data/{year}/queries_final.csv")


=== Notebook 02 started ===
SEED=42, YEARS=[2005, 2015, 2025], TYPES=['summary', 'keyword', 'function']
WORD_LIMIT=20..50, MAX_REGEN=3
LEAK_RULE: 3-gram overlap_count >= 1 => leakage
Ollama: url=http://localhost:11434/api/generate, model=gpt-oss:20b
LLM params: temperature=0.0, top_p=1.0
Working dir: c:\pydir113_bgem3
✓ Ollama is running
✓ Available models: gpt-oss:20b, gemma3:27b
[2005] sampled_patents loaded: rows=1050, cols=5
[2015] sampled_patents loaded: rows=1050, cols=5
[2025] sampled_patents loaded: rows=1050, cols=5
Stemmer: fallback_suffix_stripper, Stopwords: 55 words
[2005] resume: existing completed pairs = 0
[2015] resume: existing completed pairs = 0
[2025] resume: existing completed pairs = 0

[2005] Begin generation for 1050 patents.
[2005] checkpoint saved -> 2nd_exp\cache\query_gen\2005\checkpoints\queries_checkpoint_p50.csv (rows=150)
[2005] progress: patents=50/1050 new_rows=150
[2005] checkpoint saved -> 2nd_exp\cache\query_gen\2005\checkpoints\queries_checkpoint_

In [ ]:
import time
import numpy as np
from FlagEmbedding import BGEM3FlagModel

DEVICE = "cuda"
model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True, device=DEVICE)

# === 너의 데이터 로딩 방식에 맞춰 docs 리스트만 가져와 ===
# 예: df_p = pd.read_csv(...); docs = df_p["claims"].fillna("").astype(str).tolist()
docs = docs  # 이미 docs가 있으면 그대로

MAX_LEN = 8192
BATCH = 4              # 너가 고정하려던 값 (특히 8192에서)
N_SAMPLE = 200         # 100~500 사이 추천

sample = docs[:N_SAMPLE]

t0 = time.time()
out = model.encode(sample, batch_size=BATCH, max_length=MAX_LEN)
# out["dense_vecs"] 사용
dt = time.time() - t0

docs_per_sec = N_SAMPLE / dt
print("Dense encode throughput:", docs_per_sec, "docs/sec")

# 전체 docs에 대한 순수 인코딩 시간(초) 추정
N_TOTAL = len(docs)
est_sec = N_TOTAL / docs_per_sec
print("Estimated dense-only encoding time (sec):", est_sec)
print("Estimated dense-only encoding time (hours):", est_sec/3600)


## 아래는 보존용 코드

In [ ]:
# ============================================================
# 02_generate_queries_with_leakage_control.ipynb
# - 2005/2015/2025, 연도별 1050 샘플 특허에 대해
# - 쿼리 3유형(summary / keyword-heavy / function-oriented) 생성
# - 3-gram overlap 기반 leakage 판정 + 최대 3회 재생성
# - 결과/체크포인트를 2nd_exp/ 아래에 저장
# ============================================================

import os
import json
import re
import time
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from tqdm import tqdm

import requests

# ----------------------------
# 0) Global Settings
# ----------------------------
EXP_ROOT = Path("2nd_exp")
YEARS = [2005, 2015, 2025]

SEED = 42
np.random.seed(SEED)

# Ollama / gpt-oss (기존 코드 스타일 유지)
OLLAMA_API_URL = "http://localhost:11434/api/generate"
MODEL_NAME = "gpt-oss:20b"

# Query generation constraints (결정사항)
MIN_WORDS = 20
MAX_WORDS = 50
MAX_REGEN = 3              # leakage 걸리면 최대 3회 재생성
TEMPERATURE = 0.0          # deterministic
TOP_P = 1.0
NUM_CTX = 4096             # 필요시 조정 (프롬프트 길이 과도하면 늘려도 됨)
REQUEST_TIMEOUT = 120

# Leakage thresholds (결정사항 그대로)
LEAK_COUNT_TH = 3
LEAK_RATE_TH = 0.08

# Checkpoint
CHECKPOINT_EVERY = 25      # 25개마다 중간 저장
SLEEP_BETWEEN_CALLS = 0.0  # 로컬 ollama면 보통 0으로 둬도 됨

# ----------------------------
# 1) Utilities: Ollama
# ----------------------------
def test_ollama_connection():
    try:
        r = requests.get("http://localhost:11434", timeout=5)
        # ollama는 루트에 404를 줄 때도 있으므로 API endpoint로 한번 더
        rr = requests.post(
            OLLAMA_API_URL,
            json={"model": MODEL_NAME, "prompt": "hello", "stream": False},
            timeout=10
        )
        if rr.status_code == 200:
            print("✓ Ollama is running and API is reachable.")
            return True
        print(f"✗ Ollama API error: status={rr.status_code}, text={rr.text[:200]}")
        return False
    except Exception as e:
        print("✗ Cannot connect to Ollama. Please make sure Ollama is running.")
        print("  Start Ollama with: ollama serve")
        print("  And ensure the model exists: ollama pull gpt-oss:20b")
        print("Error:", e)
        return False

def ollama_generate(prompt: str, seed: int):
    payload = {
        "model": MODEL_NAME,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": TEMPERATURE,
            "top_p": TOP_P,
            "seed": seed,
            "num_ctx": NUM_CTX
        }
    }
    r = requests.post(OLLAMA_API_URL, json=payload, timeout=REQUEST_TIMEOUT)
    if r.status_code != 200:
        raise RuntimeError(f"Ollama API failed: {r.status_code} | {r.text[:500]}")
    out = r.json()
    return out.get("response", "").strip()

# ----------------------------
# 2) Utilities: text preprocessing + leakage (3-gram overlap)
# ----------------------------
# lightweight stopword list (NLTK 불필요)
STOPWORDS = {
    "a","an","the","and","or","but","if","then","else","when","while","for","to","of","in","on","at","by","with",
    "as","is","are","was","were","be","being","been","it","its","this","that","these","those","from","into","over",
    "under","between","within","without","via","can","could","may","might","should","would","will","shall",
    "such","than","also","using","used","use","based","including","include","includes"
}

def simple_stem(token: str) -> str:
    """
    아주 가벼운 stemming (PorterStemmer 대체, 외부 의존성 없이)
    - 완벽하지 않아도 overlap 방지 목적에는 충분.
    """
    t = token
    # common suffix strip
    for suf in ["ization","ational","fulness","ousness","iveness","tional","biliti","lessli","entli","ation","alism","aliti",
                "ousli","iviti","fulli","enci","anci","abli","izer","ator","alli","bli","ogi","li",
                "ing","edly","edly","ed","ly","es","s"]:
        if len(t) > 4 and t.endswith(suf):
            t = t[:-len(suf)]
            break
    return t

def tokenize_for_overlap(text: str):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s\-]", " ", text)  # keep ascii letters/digits/hyphen
    text = re.sub(r"\s+", " ", text).strip()
    toks = []
    for tok in text.split():
        tok = tok.strip("-")
        if not tok:
            continue
        if tok in STOPWORDS:
            continue
        tok = simple_stem(tok)
        if tok and tok not in STOPWORDS:
            toks.append(tok)
    return toks

def make_ngrams(tokens, n=3):
    if len(tokens) < n:
        return set()
    return {" ".join(tokens[i:i+n]) for i in range(len(tokens)-n+1)}

def leakage_scores(query: str, source: str):
    qtoks = tokenize_for_overlap(query)
    stoks = tokenize_for_overlap(source)
    q3 = make_ngrams(qtoks, 3)
    s3 = make_ngrams(stoks, 3)
    if len(q3) == 0:
        return 0, 0.0, set()
    inter = q3.intersection(s3)
    overlap_count = len(inter)
    overlap_rate = overlap_count / max(1, len(q3))
    return overlap_count, overlap_rate, inter

def is_leaking(overlap_count: int, overlap_rate: float) -> bool:
    return (overlap_count >= LEAK_COUNT_TH) or (overlap_rate >= LEAK_RATE_TH)

# ----------------------------
# 3) Prompt templates (고정 문구, Appendix에 그대로 넣을 수 있는 형태)
# ----------------------------
def build_prompt(query_type: str, abstract: str, claims: str, banned_ngrams=None):
    """
    query_type: 'summary' | 'keyword' | 'function'
    banned_ngrams: overlap된 3-gram을 재생성에서 금지하기 위해 제공
    """
    banned_text = ""
    if banned_ngrams:
        # 너무 길면 상위 일부만 (재생성 효율)
        banned_list = sorted(list(banned_ngrams))[:20]
        banned_text = (
            "\nBANNED PHRASES (do not reuse these exact 3-gram phrases):\n- "
            + "\n- ".join(banned_list)
            + "\n"
        )

    common_header = f"""
You will generate a single English search query to retrieve relevant patents.
STRICT REQUIREMENTS:
- Output ONLY the query text (no prefixes, no quotes, no bullet points).
- Length: between {MIN_WORDS} and {MAX_WORDS} words.
- Use general technical language; DO NOT copy wording from the patent text.
- Do NOT mention patent numbers, publication years, or that you are searching for patents.
- Avoid verbatim phrases from the source.
{banned_text}
SOURCE PATENT (for your understanding only; do NOT copy phrasing):
[ABSTRACT]
{abstract}

[CLAIMS]
{claims}
""".strip()

    if query_type == "summary":
        type_instruction = """
TASK:
Write a query that summarizes the invention at a high level, focusing on the core technical problem and the general solution concept.
Avoid listing many specific components; emphasize the invention’s main idea in general terms.
""".strip()
    elif query_type == "keyword":
        type_instruction = """
TASK:
Write a query that is component/keyword-heavy, focusing on major components, materials, and key technical terms,
but still avoid copying exact phrases from the source. Use synonyms and generalized terminology.
""".strip()
    elif query_type == "function":
        type_instruction = """
TASK:
Write a query that is function-oriented: describe what the invention does, what function it enables,
or what performance effect it achieves, in general technical terms.
Do not overly describe the structure; emphasize operation and effects.
""".strip()
    else:
        raise ValueError("Unknown query_type")

    prompt = common_header + "\n\n" + type_instruction + "\n"
    return prompt

def clean_llm_query(text: str) -> str:
    """
    LLM이 가끔 'Query:' 같은 prefix를 붙이거나 따옴표를 붙이는 경우 제거
    """
    if not text:
        return ""
    t = text.strip()

    # remove common prefixes
    prefixes = [
        "Search Query:", "Query:", "Here is", "The search query is:", "A possible search query:",
        "Generated query:", "Here's", "Suggested query:", "My query:"
    ]
    for p in prefixes:
        if t.lower().startswith(p.lower()):
            t = t[len(p):].strip()

    t = t.strip().strip('"').strip("'").strip()
    # collapse whitespace
    t = re.sub(r"\s+", " ", t).strip()
    return t

def word_count(text: str) -> int:
    return len([w for w in text.split() if w.strip()])

# ----------------------------
# 4) Generation with leakage control
# ----------------------------
def generate_query_for_patent(query_type: str, abstract: str, claims: str, base_seed: int):
    """
    returns:
      query_text, leak_flag, overlap_count, overlap_rate, regen_attempts
    """
    source = (abstract or "") + "\n" + (claims or "")
    banned = set()
    last_q = ""
    last_stats = (0, 0.0)

    for attempt in range(1, MAX_REGEN + 1):
        prompt = build_prompt(query_type, abstract, claims, banned_ngrams=banned if attempt > 1 else None)

        # attempt별로 seed를 약간 변경하되 deterministic 유지
        seed = base_seed + attempt

        raw = ollama_generate(prompt, seed=seed)
        q = clean_llm_query(raw)

        # length enforcement: 너무 짧거나 길면 재생성
        wc = word_count(q)
        if wc < MIN_WORDS or wc > MAX_WORDS:
            # 길이 위반도 재생성 사유로 간주
            last_q = q
            last_stats = (0, 0.0)
            # banned는 유지(없을 수 있음)
            continue

        oc, orate, inter = leakage_scores(q, source)
        last_q = q
        last_stats = (oc, orate)

        if not is_leaking(oc, orate):
            return q, 0, oc, orate, attempt  # leak_flag=0

        # leaking이면 overlap된 3-gram 일부를 banned에 추가 후 재시도
        banned.update(inter)

        if SLEEP_BETWEEN_CALLS > 0:
            time.sleep(SLEEP_BETWEEN_CALLS)

    # 여기 오면 MAX_REGEN 내에 leakage-free 실패
    oc, orate = last_stats
    return last_q, 1, oc, orate, MAX_REGEN  # leak_flag=1 (끝까지 해결 못함)

# ----------------------------
# 5) Main runner (year-wise; checkpoints)
# ----------------------------
TYPE_MAP = {
    "summary": "T1_summary",
    "keyword": "T2_keyword",
    "function": "T3_function"
}

def ensure_dirs(year: int):
    base = EXP_ROOT / "queries" / str(year)
    (base / "checkpoints").mkdir(parents=True, exist_ok=True)
    (base / "queries_by_type").mkdir(parents=True, exist_ok=True)
    return base

def load_year_sample(year: int):
    year_dir = EXP_ROOT / "data" / str(year)
    meta_path = year_dir / "meta.json"
    samp_path = year_dir / "sampled_patents.csv"
    if not meta_path.exists():
        raise FileNotFoundError(f"Missing meta.json: {meta_path} (run Notebook 01 first)")
    if not samp_path.exists():
        raise FileNotFoundError(f"Missing sampled_patents.csv: {samp_path} (run Notebook 01 first)")

    meta = json.load(open(meta_path, "r", encoding="utf-8"))
    df = pd.read_csv(samp_path)

    # column names
    id_col = meta["id_col"]
    claims_col = meta["claims_col"]
    abstract_col = meta["abstract_col"]

    # safety normalize
    df[id_col] = df[id_col].astype(str)
    df[claims_col] = df[claims_col].fillna("").astype(str)
    df[abstract_col] = df[abstract_col].fillna("").astype(str)

    return df, meta

def run_for_year(year: int):
    base = ensure_dirs(year)
    df, meta = load_year_sample(year)

    id_col = meta["id_col"]
    claims_col = meta["claims_col"]
    abstract_col = meta["abstract_col"]

    out_all = base / "queries_all.csv"
    # resume if exists
    if out_all.exists():
        done = pd.read_csv(out_all)
        done_keys = set(zip(done[id_col].astype(str), done["query_type"]))
        print(f"[{year}] Resuming: existing queries_all.csv found ({len(done)} rows).")
    else:
        done = None
        done_keys = set()

    rows = []
    stats = {
        "year": year,
        "n_patents": int(len(df)),
        "started_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "leak_fail_count": 0,
        "regen_mean": None,
        "per_type": {k: {"n": 0, "leak_fail": 0, "regen_sum": 0} for k in TYPE_MAP.keys()}
    }

    pbar = tqdm(df.itertuples(index=False), total=len(df), desc=f"QueryGen {year}")
    processed = 0
    for r in pbar:
        pid = str(getattr(r, id_col))
        abstract = getattr(r, abstract_col)
        claims = getattr(r, claims_col)

        for qt in TYPE_MAP.keys():
            # skip if already done
            if (pid, qt) in done_keys:
                continue

            base_seed = (hash(f"{year}|{pid}|{qt}|{SEED}") % 10_000_000)

            qtext, leak_flag, oc, orate, attempts = generate_query_for_patent(
                query_type=qt,
                abstract=abstract,
                claims=claims,
                base_seed=base_seed
            )

            row = {
                "year": year,
                id_col: pid,
                "query_type": qt,
                "query_type_label": TYPE_MAP[qt],
                "query_text": qtext,
                "word_count": word_count(qtext),
                "leak_flag": leak_flag,
                "overlap_count_3gram": oc,
                "overlap_rate_3gram": orate,
                "regen_attempts": attempts,
                "seed_base": base_seed
            }
            rows.append(row)

            # stats
            stats["per_type"][qt]["n"] += 1
            stats["per_type"][qt]["regen_sum"] += attempts
            if leak_flag == 1:
                stats["leak_fail_count"] += 1
                stats["per_type"][qt]["leak_fail"] += 1

            processed += 1

            # checkpoint
            if processed % CHECKPOINT_EVERY == 0:
                ck = pd.DataFrame(rows)
                ck_path = base / "checkpoints" / f"ckpt_{processed:06d}.csv"
                ck.to_csv(ck_path, index=False)

                # append to queries_all
                if out_all.exists():
                    ck.to_csv(out_all, mode="a", index=False, header=False)
                else:
                    ck.to_csv(out_all, index=False)

                # update done_keys
                for rr in rows:
                    done_keys.add((rr[id_col], rr["query_type"]))
                rows = []  # reset buffer

                pbar.set_postfix({"saved": processed, "leak_fail": stats["leak_fail_count"]})

    # flush remaining
    if rows:
        ck = pd.DataFrame(rows)
        if out_all.exists():
            ck.to_csv(out_all, mode="a", index=False, header=False)
        else:
            ck.to_csv(out_all, index=False)
        for rr in rows:
            done_keys.add((rr[id_col], rr["query_type"]))
        rows = []

    # finalize stats
    # load all for this year
    all_df = pd.read_csv(out_all)
    stats["completed_at"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    stats["n_queries_total"] = int(len(all_df))
    stats["leak_fail_rate"] = float(all_df["leak_flag"].mean()) if len(all_df) else None
    stats["regen_mean"] = float(all_df["regen_attempts"].mean()) if len(all_df) else None

    stats_path = base / "stats_querygen.json"
    with open(stats_path, "w", encoding="utf-8") as f:
        json.dump(stats, f, indent=2, ensure_ascii=False)

    print(f"[{year}] Saved queries_all.csv: {out_all}")
    print(f"[{year}] Saved stats: {stats_path}")

    # split by type
    by_type_dir = base / "queries_by_type"
    for qt in TYPE_MAP.keys():
        sub = all_df[all_df["query_type"] == qt].copy()
        sub_path = by_type_dir / f"{TYPE_MAP[qt]}.csv"
        sub.to_csv(sub_path, index=False)
    print(f"[{year}] Saved type-split files under: {by_type_dir}")

# ----------------------------
# 6) Run
# ----------------------------
print("Testing Ollama connection...")
ollama_ok = test_ollama_connection()
if not ollama_ok:
    raise RuntimeError("Ollama not available. Start 'ollama serve' and ensure the model is installed.")

for y in YEARS:
    run_for_year(y)

print("\n✅ Notebook 02 finished. Next: Notebook 03 (Aligned 2-way hybrid weighted-RRF comparison).")
